<a href="https://colab.research.google.com/github/profcomff/chatbot-mark-api/blob/dev_fedor/notebooks/Create_db.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Нужно обновить путь до database.xslx

In [ ]:
# !rm -rf /content/chatbot-mark-api

In [15]:
# !git clone https://github.com/profcomff/chatbot-mark-api.git
!git clone --branch dev_fedor https://github.com/profcomff/chatbot-mark-api.git

fatal: destination path 'chatbot-mark-api' already exists and is not an empty directory.


# Библиотеки


In [16]:
!pip install langchain transformers sentence-transformers -q
!pip install -U langchain-community -q
!pip install -qU "langchain-chroma>=0.1.2" -q
!pip install langchain_huggingface -q

In [17]:
from tqdm import tqdm

import numpy as np
import pandas as pd

from transformers import XLMRobertaTokenizer, XLMRobertaModel
import torch

from langchain.schema import Document

from langchain_chroma import Chroma

import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

# Функции/классы

In [18]:
import sys
sys.path.append("/content/chatbot-mark-api")

from nn.search import E5LangChainEmbedder

In [19]:
def safe_add_documents(vector_store, chunks, chroma_batch_size=1000):
    with tqdm(total=len(chunks), desc="Добавление в Chroma", unit="doc") as pbar:
        for i in range(0, len(chunks), chroma_batch_size):
            try:
                batch = chunks[i:i+chroma_batch_size]
                vector_store.add_documents(batch)
                pbar.update(len(batch))
            except Exception as e:
                if "Batch size" in str(e) and "greater than max" in str(e):
                    new_size = chroma_batch_size // 2
                    print(f"Ошибка: {e}. Уменьшаю размер батча до {new_size}")
                    return safe_add_documents(vector_store, chunks[i:], new_size)
                raise
            finally:
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
    print("Все документы успешно добавлены!")

# 0. Загрузка контекстов / скачивание модели


In [20]:
# answers = pd.read_excel('/content/chatbot-mark-api/file/database_v2.xlsx') #!!! should change
answers = pd.read_excel('/content/chatbot-mark-api/file/database_v2_key_words.xlsx')

display(answers.answer[0])
display(answers.head(2))

'Карта зачет. https://vk.com/wall-24234717_22977\nЭто ваш профсоюзный билет. С помощью этой карты вы можете получать скидки у полезных для студентов популярных брендов, участвовать в конкурсах и розыгрышах, а также посещать концерты и мероприятия. \nПолный перечень скидок есть в статье: vk.cc/bYSCNw.'

,Unnamed: 0,topic_name,answer,id,Key words
0,0,Карта зачет,Карта зачет. https://vk.com/wall-24234717_2297...,0,Карта зачет
1,1,Как вступить в профсоюз,Как вступить в профсоюз? Чтобы вступить в Проф...,1,Профсоюз


link to model in HuggingFace [e5-base-en-ru](https://huggingface.co/d0rj/e5-base-en-ru)

In [21]:
tokenizer = XLMRobertaTokenizer.from_pretrained("d0rj/e5-base-en-ru", use_cache=False)
search_model = XLMRobertaModel.from_pretrained("d0rj/e5-base-en-ru", use_cache=False)

# 1. Создание БД c помощью e5


In [22]:
all_chunks = []

for answer, topic_name, id_val, kw in zip(answers['answer'], answers['topic_name'], answers['id'], answers['Key words']):
    all_chunks.append(Document(
        page_content=answer,
        metadata={
            "source": topic_name.strip(),
            "id": id_val,
            "key_words": kw,
        }
    ))

# Инициализация эмбеддера E5
embedder = E5LangChainEmbedder(
    tokenizer=tokenizer,
    model=search_model,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    add_prefix=True,  #!!!
    disable_tqdm=False,
)

# Создание или загрузка векторного хранилища Chroma
vector_store = Chroma(
    collection_name="docs",
    embedding_function=embedder,
    persist_directory="./chroma_db"  #!!!
)

# Безопасное добавление документов в векторное хранилище
safe_add_documents(vector_store, all_chunks)

Добавление в Chroma: 100%|██████████| 100/100 [02:16<00:00,  1.36s/doc]

Все документы успешно добавлены!


In [26]:
!zip -r chroma_db.zip chroma_db/

  adding: chroma_db/ (stored 0%)
  adding: chroma_db/480e3841-dae8-45fa-a207-765cce1eee47/ (stored 0%)
  adding: chroma_db/480e3841-dae8-45fa-a207-765cce1eee47/data_level0.bin (deflated 100%)
  adding: chroma_db/480e3841-dae8-45fa-a207-765cce1eee47/header.bin (deflated 61%)
  adding: chroma_db/480e3841-dae8-45fa-a207-765cce1eee47/length.bin (deflated 100%)
  adding: chroma_db/480e3841-dae8-45fa-a207-765cce1eee47/link_lists.bin (stored 0%)
  adding: chroma_db/chroma.sqlite3 (deflated 50%)


# Подключение БД

In [23]:
vector_store = Chroma(
    collection_name="docs",
    embedding_function=embedder,
    persist_directory="./chroma_db"
)

In [24]:
query = "карта зачет профком?"

relevant_docs = vector_store.similarity_search(
    query,
    k=3,
)

In [25]:
relevant_docs

[Document(id='8d31c2b1-2ced-477f-aa89-bbe28a0e21c2', metadata={'id': 16, 'source': 'Проблемы с картой Зачет'}, page_content='Проблемы с картой Зачет. Если вам не выдали карту Зачёт, обратитесь в Профком (к. 2-39). Проверьте, что в личном кабинете члена профсоюза (https://lk.msuprof.com/) загружена фотография. \nВ случае утери карты Зачёт, необходимо обратиться в Профком.'),
 Document(id='b72657f1-f565-4154-915b-e3f87bdac1b7', metadata={'id': 16, 'key_words': 'Карта  зачет', 'source': 'Проблемы с картой Зачет'}, page_content='Проблемы с картой Зачет. Если вам не выдали карту Зачёт, обратитесь в Профком (к. 2-39). Проверьте, что в личном кабинете члена профсоюза (https://lk.msuprof.com/) загружена фотография. \nВ случае утери карты Зачёт, необходимо обратиться в Профком.'),
 Document(id='b67a0a4e-4d50-45e0-b187-b073f189727c', metadata={'source': 'Карта зачет', 'id': 0}, page_content='Карта зачет. https://vk.com/wall-24234717_22977\nЭто ваш профсоюзный билет. С помощью этой карты вы может